# Week 14 — Modern Generative Models: demonstrations
**Machine Learning I (MATH 60629A)** — Laurent Charlin

This notebook contains the three demonstrations from class. Each section is independent.

| Demo | Idea | Hardware |
|---|---|---|
| A | A GMM is already a generative model: *evaluate* and *sample* | CPU |
| B | A 2D diffusion model, from scratch | CPU (~1 min) |
| C | Text → image and *guidance* (classifier-free guidance) | **GPU** (Runtime → Change runtime type → T4 GPU) |

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import make_moons

rng = np.random.default_rng(0)

# Toy data used in demos A and B: two "moons" in 2D
X, _ = make_moons(n_samples=5000, noise=0.05, random_state=0)
X = (X - X.mean(0)) / X.std(0)          # standardize

def frame(ax, title):
    ax.set_xlim(-3, 3); ax.set_ylim(-3, 3); ax.set_aspect("equal")
    ax.set_xticks([]); ax.set_yticks([]); ax.set_title(title)

fig, ax = plt.subplots(figsize=(4, 4))
ax.scatter(*X[:2000].T, s=3); frame(ax, "Our data")
plt.show()

---
## Demo A — You have already built a generative model

In week 8, we used Gaussian mixture models (GMMs) for **clustering**.
A GMM defines a distribution $P(x)$, so we can use it to:

1. **Evaluate**: what is the (log-)likelihood of a point? → `score_samples`
2. **Sample**: produce *new* points → `sample`

In [ ]:
from sklearn.mixture import GaussianMixture

gmm = GaussianMixture(n_components=6, random_state=0).fit(X)
new_points, _ = gmm.sample(2000)

from matplotlib.patches import Ellipse

def ellipse(ax, mu, cov, weight, n_std=2):
    # ellipse axes = eigenvectors of the covariance; lengths = n_std standard deviations
    vals, vecs = np.linalg.eigh(cov)
    angle = np.degrees(np.arctan2(vecs[1, 1], vecs[0, 1]))
    l, h = 2 * n_std * np.sqrt(vals[::-1])
    ax.add_patch(Ellipse(mu, l, h, angle=angle, fill=False, lw=2, ec="navy", alpha=0.3 + 0.7 * weight))
    ax.plot(*mu, "+", c="navy", ms=10, mew=2)

fig, axes = plt.subplots(1, 3, figsize=(14, 4.5))
axes[0].scatter(*X[:2000].T, s=3); frame(axes[0], "Real data")
axes[1].scatter(*X[:2000].T, s=3, c="lightgrey")
for k in range(gmm.n_components):
    ellipse(axes[1], gmm.means_[k], gmm.covariances_[k], gmm.weights_[k] / gmm.weights_.max())
frame(axes[1], "Learned components (mean, covariance)")
axes[2].scatter(*new_points.T, s=3, c="C1"); frame(axes[2], "Samples from the GMM")
plt.show()

In [ ]:
# 1) Evaluate: a point "on the moon" vs a point "in the gap"
points = np.array([[0.0, 1.4],     # on the upper moon
                   [0.0, 0.0]])    # between the two moons
print("log P(x):", gmm.score_samples(points).round(2))

Each ellipse is centred on the mean $\mu_k$ of a component; its shape comes from the covariance $\Sigma_k$ (2 standard deviations). The darker the line, the larger the weight $\pi_k$.

**Try:** change `n_components` (2, 6, 8, 50). What happens?

**Takeaway:** for 2D points, a GMM is enough. For an image of 256×256×3 ≈ 200,000 strongly dependent numbers, we need neural networks. That is the rest of the class.

---
## Demo B — A 2D diffusion model, from scratch

The idea in three sentences:
1. **Destroying is easy**: add Gaussian noise, little by little, until only pure noise is left.
2. **Learn to undo it**: a network receives a noisy point $x_t$ and the step $t$, and **predicts the noise** that was added. It is a plain **regression** (squared loss).
3. **Generate**: start from pure noise and remove the predicted noise, step by step.

Adding noise has a closed form: $x_t = \sqrt{\bar\alpha_t}\, x_0 + \sqrt{1-\bar\alpha_t}\,\varepsilon$, with $\varepsilon \sim \mathcal N(0, I)$.

In [ ]:
import torch, torch.nn as nn
torch.manual_seed(0)

T = 100                                        # number of noising steps
betas = torch.linspace(1e-4, 0.1, T)           # amount of noise added at each step
alphas = 1 - betas
alpha_bar = torch.cumprod(alphas, dim=0)       # how much of the original signal is left at step t

X_t = torch.tensor(X, dtype=torch.float32)

def add_noise(x0, t, eps):
    # x_t = sqrt(alpha_bar_t) * x0 + sqrt(1 - alpha_bar_t) * eps
    ab = alpha_bar[t].unsqueeze(1)
    return ab.sqrt() * x0 + (1 - ab).sqrt() * eps

# The "forward" process: watch the data dissolve
steps = [0, 10, 25, 50, 99]
fig, axes = plt.subplots(1, 5, figsize=(17, 3.6))
for ax, t in zip(axes, steps):
    tt = torch.full((2000,), t)
    xt = add_noise(X_t[:2000], tt, torch.randn(2000, 2))
    ax.scatter(*xt.T, s=2); frame(ax, f"t = {t}")
plt.suptitle("Forward process: data → noise"); plt.show()

In [ ]:
# The "denoiser" network: a small MLP that receives (x_t, t) and predicts the noise epsilon
class Denoiser(nn.Module):
    def __init__(self, h=128, n_freq=8):
        super().__init__()
        self.freq = torch.arange(1, n_freq + 1) * torch.pi
        self.net = nn.Sequential(
            nn.Linear(2 + 2 * n_freq, h), nn.ReLU(),
            nn.Linear(h, h), nn.ReLU(),
            nn.Linear(h, h), nn.ReLU(),
            nn.Linear(h, 2))
    def forward(self, x, t):
        # encode the step t with sines/cosines (like the positional encoding of transformers)
        tt = (t.float() / T).unsqueeze(1) * self.freq
        return self.net(torch.cat([x, tt.sin(), tt.cos()], dim=1))

model = Denoiser()
opt = torch.optim.Adam(model.parameters(), lr=1e-3)

# Training: an ordinary regression!
losses = []
for it in range(4000):
    x0 = X_t[torch.randint(0, len(X_t), (512,))]
    t = torch.randint(0, T, (512,))
    eps = torch.randn_like(x0)
    loss = ((model(add_noise(x0, t, eps), t) - eps) ** 2).mean()   # || eps - eps_predicted ||^2
    opt.zero_grad(); loss.backward(); opt.step()
    losses.append(loss.item())

plt.plot(np.convolve(losses, np.ones(50) / 50, mode="valid"))
plt.xlabel("iteration"); plt.ylabel("loss (moving average)"); plt.show()

In [ ]:
# Generation: start from pure noise and denoise step by step
@torch.no_grad()
def generate(n=2000, keep=(99, 50, 25, 10, 0)):
    x = torch.randn(n, 2)
    snapshots = {T - 1: x.clone()}
    for t in reversed(range(T)):
        tt = torch.full((n,), t)
        eps_pred = model(x, tt)
        # remove the predicted share of noise
        x = (x - betas[t] / (1 - alpha_bar[t]).sqrt() * eps_pred) / alphas[t].sqrt()
        if t > 0:                                   # a little randomness at each step
            x = x + betas[t].sqrt() * torch.randn_like(x)
        if t in keep:
            snapshots[t] = x.clone()
    return snapshots

snaps = generate()
fig, axes = plt.subplots(1, 5, figsize=(17, 3.6))
for ax, t in zip(axes, [99, 50, 25, 10, 0]):
    ax.scatter(*snaps[t].T, s=2, c="C1"); frame(ax, f"t = {t}")
plt.suptitle("Reverse process: noise → data"); plt.show()

In [ ]:
# (Optional) Animation of the reverse process
from matplotlib import animation
from IPython.display import HTML

@torch.no_grad()
def trajectory(n=1500):
    x = torch.randn(n, 2); traj = [x.clone()]
    for t in reversed(range(T)):
        eps_pred = model(x, torch.full((n,), t))
        x = (x - betas[t] / (1 - alpha_bar[t]).sqrt() * eps_pred) / alphas[t].sqrt()
        if t > 0: x = x + betas[t].sqrt() * torch.randn_like(x)
        traj.append(x.clone())
    return traj

traj = trajectory()
fig, ax = plt.subplots(figsize=(4.5, 4.5)); sc = ax.scatter(*traj[0].T, s=2, c="C1")
def update(i):
    sc.set_offsets(traj[i].numpy()); frame(ax, f"step {i} / {T}"); return (sc,)
anim = animation.FuncAnimation(fig, update, frames=len(traj), interval=60)
plt.close(fig); HTML(anim.to_jshtml())

**Try:**
- Reduce `T` to 10. Is generation as good? (A few big steps are harder than many small ones.)
- Replace `make_moons` with another shape (e.g., `sklearn.datasets.make_circles`) and retrain.

**Why do the moons only appear at the end?** The moons are thin (about 0.06 to 0.1 thick in our units). We can only see them once the remaining noise is smaller than that thickness. With our schedule, the standard deviation of the remaining noise is still about 0.86 at $t = 50$, 0.53 at $t = 25$ and 0.23 at $t = 10$: during the first half of the reverse process, the cloud looks almost like $\mathcal N(0, I)$, and the model can only move mass toward two broad regions. The thin curves only appear in the last ten or so steps. Diffusion builds the coarse structure first, then the fine details. (A "cosine" schedule, Nichol & Dhariwal 2021, spreads the steps more evenly.)

**Takeaway:** the network has never seen "a moon". It only learned to remove noise. Image generation models do *exactly* this, with a bigger network (U-Net or transformer) and images instead of points.

---
## Demo C — Text → image and guidance (*classifier-free guidance*)

⚠️ **Requires a GPU.** (Runtime → Change runtime type → T4 GPU.) The first model download takes a few minutes.

We use Stable Diffusion 1.5, a **latent diffusion** model: denoising happens in a space compressed by an auto-encoder, then the decoder produces the image.

**Guidance** $w$ is a dial: the model predicts the noise *with* and *without* the text, and we push in the direction of the text:
$\hat\varepsilon = \varepsilon_\theta(x_t, \varnothing) + w\,\big(\varepsilon_\theta(x_t, \text{text}) - \varepsilon_\theta(x_t, \varnothing)\big)$

In [ ]:
!pip -q install diffusers transformers accelerate

In [ ]:
import torch
from diffusers import StableDiffusionPipeline

pipe = StableDiffusionPipeline.from_pretrained(
    "stable-diffusion-v1-5/stable-diffusion-v1-5", torch_dtype=torch.float16
).to("cuda")
pipe.set_progress_bar_config(disable=True)

In [ ]:
prompt = "a watercolor painting of a fox reading a book in a library"
seed = 0
guidances = [1.0, 3.0, 7.5, 15.0]      # w = 1: no guidance

images = []
for w in guidances:
    g = torch.Generator("cuda").manual_seed(seed)   # same seed → same starting noise
    images.append(pipe(prompt, guidance_scale=w, num_inference_steps=30, generator=g).images[0])

fig, axes = plt.subplots(1, len(images), figsize=(16, 4.5))
for ax, im, w in zip(axes, images, guidances):
    ax.imshow(im); ax.set_title(f"guidance w = {w}"); ax.axis("off")
plt.show()

**Questions:** what changes as $w$ increases? Does the image follow the text more closely? Is it more or less varied? More natural or more "saturated"?

In [ ]:
# Watch the denoising: decode the latent image at a few intermediate steps
intermediates = []
def callback(pipe, i, t, kw):
    if i % 6 == 0 or i == 29:
        with torch.no_grad():
            lat = kw["latents"] / pipe.vae.config.scaling_factor
            img = pipe.vae.decode(lat).sample
        img = ((img[0].float().clamp(-1, 1) + 1) / 2).permute(1, 2, 0).cpu().numpy()
        intermediates.append((i, img))
    return kw

g = torch.Generator("cuda").manual_seed(seed)
pipe(prompt, guidance_scale=7.5, num_inference_steps=30, generator=g,
     callback_on_step_end=callback, callback_on_step_end_tensor_inputs=["latents"])

fig, axes = plt.subplots(1, len(intermediates), figsize=(18, 3.6))
for ax, (i, img) in zip(axes, intermediates):
    ax.imshow(img); ax.set_title(f"step {i + 1}"); ax.axis("off")
plt.show()

In [ ]:
# (For the instructor) Save the generated images, e.g. for the slides
for w, im in zip(guidances, images):
    im.save(f"guidance_w{w}.png")

**Try:** keep the same seed and change a single word of the prompt. Then keep the prompt and change the seed.

**Takeaway:** the text is just a *condition* given to the denoiser; guidance sets the trade-off **faithfulness to the text ↔ diversity**.